# Exercices 8

[Télécharger l'exercice](../08_exercice.zip)

# Exercice: Température de la croûte terrestre

Nous allons modéliser l'évolution du géotherme de la croûte terrestre en deux dimensions en incluant un paramètre d'exhumation et en simulant finalement une faille active dans un domaine modélisé de 50 km de longueur et de 25 km de profondeur (Figure 1). L'évolution du champ de température $T$ en 2D est le résultat de la diffusion thermique le long des axes $x$ et $z$, ainsi que de l'advection par exhumation, $\dot{e}$, le long d'une seule dimension ($z$) :

$$\frac{\partial T}{\partial t} = -\frac{\partial q_x}{\partial x} -\frac{\partial q_z}{\partial z}  - \dot{e} \frac{\partial T}{\partial z}\ ,\: $$

$$q_x = - D \frac{\partial T}{\partial x} \: ; \: \: q_z = - D \frac{\partial T}{\partial z}$$

La température initiale varie linéairement entre les valeurs données en surface et au fond du domaine modélisé. Tous les paramètres sont donnés dans la Table 1.

Puisque l'exhumation des roches s'effectue dans la seule direction $z$ (verticale), nous n'avons besoin de résoudre l'advection que dans la dimension $z$. Pour cela, il faut créer une matrice contenant les vitesses `Vz` de taille `(nz-1, nx)`. Ceci afin que les vitesses se situent à l'interface entre deux cellules de température `T`, par analogie avec les flux.

![](./fig/ex_1.png)

*Les trois variantes du modèle. Le fond coloré montre le géotherme initial, les flèches blanches l'exhumation, et les flèches rouges le flux de chaleur imposé à la base en Q3.*

Deux dimensions compliquent aussi le calcul de `dt`. La contrainte de diffusion prend en compte le plus petit des deux pas d'espace `dx` et `dz`, et la contrainte d'advection porte sur la direction `z` (la seule où il y a advection) :

```python
dt_diff = min(dx, dz)**2 / (4.1 * D)          # contrainte de la diffusion
dt_adv  = dz / (2.1 * np.max(np.abs(Vz)))     # contrainte de l'advection
dt      = min(dt_max, dt_diff, dt_adv)        # pas de temps retenu
```
Vous noterez que la valeur maximale de la matrice des vitesses `Vz` en valeur absolue est donnée par `np.max(np.abs(Vz))`.

Nous voulons écrire un code avec trois variantes. Définissez un paramètre Q, qui vaut 1, 2 ou 3, et utilisez des clauses conditionnelles pour écrire un unique code générique qui fonctionne pour les trois variantes.

- Q1: Dans une première version du code, la condition de température en profondeur est fixée à 700°C partout mais le taux d'exhumation change entre 5 km/Myr dans la partie gauche et 15 km/Myr dans la partie droite. Le flux de chaleur au travers des bords latéraux est nul. Le scénario tectonique équivalent serait la présence d'une faille normale séparant deux domaines s'érodant à des taux différents (Figure ci-dessus).

- Q2: Dans une deuxième version du code, utilisez une condition aux bords de température $T$ en profondeur qui varie le long de l'axe $x$. La moitié gauche à 700°C et la droite à 900°C. La situation tectonique correspondante serait la subduction d'une croûte océanique froide sous la moitié gauche du modèle. Le flux de chaleur au travers des bords latéraux est également nul.

- Q3: Jusqu'ici la température en profondeur était **imposée** (condition de Dirichlet). Dans cette troisième version, nous imposons à la place le **flux de chaleur géothermique** $q_\mathrm{base}$ qui entre par le bas du domaine: c'est une condition de Neumann **non-nulle**. La température à 25 km de profondeur n'est alors plus une donnée, elle devient un **résultat** du modèle. Pour isoler l'effet de cette nouvelle condition aux bords, on supprime l'exhumation ($\dot{e} = 0$) dans cette question.

  Rappelons que la loi de Fourier relie le flux au gradient, $q = -D \frac{\partial T}{\partial z}$, d'où le gradient à imposer à la base:

  $$\frac{\partial T}{\partial z}(\mathrm{base}) = -\frac{q_\mathrm{base}}{D}.$$

  À la fin de la simulation, relevez la température atteinte à 25 km de profondeur et comparez-la aux 700°C imposés en Q1. Vous pouvez vérifier votre résultat avec la solution analytique à l'équilibre, $T_\mathrm{bas} = T_\mathrm{haut} + \frac{q_\mathrm{base}}{D} L_z$.

| **Paramètre**                                  | **Valeur** | **Unité** |
|------------------------------------------------|------------|-----------|
| Diffusivité de la température, $D$             | 150        | km²/Myr   |
| Température à la surface, $T_\mathrm{haut}$    | 25         | °C        |
| Profondeur du modèle, $L_z$                    | 25 000     | m         |
| Largeur du modèle, $L_x$                       | 50         | km        |
| Nombre de nœuds en $x$, $n_x$                  | 50         | —         |
| Nombre de nœuds en $z$, $n_z$                  | 60         | —         |
| Durée totale                                   | 10         | Myr       |
| **Q=1**                                        |            |           |
| T à 25 km de profondeur                        | 700        | °C        |
| Taux d'exhumation entre 0 et 25 km             | 5          | km/Myr    |
| Taux d'exhumation entre 25 et 50 km            | 15         | km/Myr    |
| **Q=2**                                        |            |           |
| T à 25 km de profondeur entre $x = 0-25$ km    | 700        | °C        |
| T à 25 km de profondeur entre $x = 25-50$ km   | 900        | °C        |
| Taux d'exhumation, $\dot{e}$                   | 10         | km/Myr    |
| **Q=3**                                        |            |           |
| Flux géothermique à la base, $q_\mathrm{base}$ | 4500       | °C·km/Myr |
| Taux d'exhumation, $\dot{e}$                   | 0          | km/Myr    |

> ⚠️ **Attention aux unités !** Les paramètres ne vous sont pas tous donnés dans des unités compatibles entre elles. Convertissez-les explicitement dans votre code, avant la boucle temporelle.


*Table 1: Paramètres du modèle.*

### ✅ **À vous de faire !**